# Taxi Trips - Distribution & Outlier Diagnostics
1. **Distribution plots** for every variable
2. **Outlier diagnostic plots** (short/long trips, extreme distances/fares, geo outliers, consistency checks)

> Plots are displayed inline below.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

BASE = Path(os.getcwd()).parent
PARQUET = BASE / "data" / "Taxi_Trips_compact.parquet"

print("Loading data...")
df = pd.read_parquet(PARQUET)
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
df.info()

---
## Part 1: Variable Distributions
One plot per variable. For large datasets we use a sample of 500k for histograms.

In [ ]:
numeric_cols = [
    "trip_seconds", "trip_miles", "fare_usd", "tips_usd", "tolls_usd",
    "extras_usd", "trip_total_usd", "pickup_lat", "pickup_lon",
    "dropoff_lat", "dropoff_lon", "pickup_community_area", "dropoff_community_area"
]
categorical_cols = ["payment_type", "company"]
datetime_cols = ["trip_start", "trip_end"]

SAMPLE_N = 500_000
np.random.seed(42)
sample_idx = np.random.choice(len(df), size=min(SAMPLE_N, len(df)), replace=False)
dfs = df.iloc[sample_idx]
print(f"Sample size for histograms: {len(dfs):,}")

### Numeric Variables

In [ ]:
for col in numeric_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    vals = dfs[col].dropna()

    axes[0].hist(vals, bins=100, edgecolor="white", linewidth=0.3, color="steelblue")
    axes[0].set_title(f"{col} - Full Range")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Count")

    p1, p99 = vals.quantile(0.01), vals.quantile(0.99)
    if p1 < p99:
        clipped = vals[(vals >= p1) & (vals <= p99)]
        axes[1].hist(clipped, bins=100, edgecolor="white", linewidth=0.3, color="darkorange")
        axes[1].set_title(f"{col} - 1st to 99th Pctl [{p1:.2f} - {p99:.2f}]")
        axes[1].set_xlabel(col)
        axes[1].set_ylabel("Count")
    else:
        axes[1].text(0.5, 0.5, "No range to zoom", ha="center", transform=axes[1].transAxes)

    fig.suptitle(f"Distribution of {col}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"  {col}: min={vals.min():.2f}, max={vals.max():.2f}, median={vals.median():.2f}")

### Categorical Variables

In [ ]:
for col in categorical_cols:
    counts = df[col].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    top = counts.head(25) if len(counts) > 25 else counts
    axes[0].barh(range(len(top)), top.values, color="steelblue")
    axes[0].set_yticks(range(len(top)))
    axes[0].set_yticklabels([str(v)[:30] for v in top.index], fontsize=8)
    axes[0].set_title(f"{col} - {'Top 25' if len(counts) > 25 else 'All Categories'}")
    axes[0].set_xlabel("Count")
    axes[0].invert_yaxis()

    top10 = counts.head(10)
    if len(counts) > 10:
        top10 = pd.concat([top10, pd.Series({"Other": counts.iloc[10:].sum()})])
    axes[1].pie(top10.values, labels=[str(v)[:25] for v in top10.index],
                autopct="%1.1f%%", pctdistance=0.85, startangle=90,
                textprops={"fontsize": 7})
    axes[1].set_title(f"{col} - Share")

    fig.suptitle(f"Distribution of {col}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"  {col}: {len(counts)} categories")

### Datetime Variables

In [ ]:
for col in datetime_cols:
    dt = df[col].dropna()
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    daily = dt.dt.date.value_counts().sort_index()
    axes[0, 0].plot(pd.to_datetime(daily.index), daily.values, linewidth=0.5, color="steelblue")
    axes[0, 0].set_title(f"{col} - Trips per Day")
    axes[0, 0].set_ylabel("Count")
    axes[0, 0].tick_params(axis="x", rotation=45)

    hourly = dt.dt.hour.value_counts().sort_index()
    axes[0, 1].bar(hourly.index, hourly.values, color="darkorange", edgecolor="white")
    axes[0, 1].set_title(f"{col} - By Hour of Day")
    axes[0, 1].set_xlabel("Hour")
    axes[0, 1].set_ylabel("Count")

    dow = dt.dt.dayofweek.value_counts().sort_index()
    day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    axes[1, 0].bar([day_names[i] for i in dow.index], dow.values, color="seagreen", edgecolor="white")
    axes[1, 0].set_title(f"{col} - By Day of Week")
    axes[1, 0].set_ylabel("Count")

    monthly = dt.dt.to_period("M").value_counts().sort_index()
    axes[1, 1].bar([str(p) for p in monthly.index], monthly.values, color="mediumpurple", edgecolor="white")
    axes[1, 1].set_title(f"{col} - By Month")
    axes[1, 1].set_ylabel("Count")
    axes[1, 1].tick_params(axis="x", rotation=90, labelsize=6)

    fig.suptitle(f"Temporal Distribution of {col}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Part 2: Outlier Diagnostics
Visual exploration of potential outliers. These are **counted and documented**, not filtered.

### 2a: Very Short (< 60s) & Very Long Trips (duration)
- Short trips: `trip_seconds < 60`
- Long trips: `trip_seconds > 99th percentile`

In [ ]:
ts = df["trip_seconds"].dropna()
p99 = ts.quantile(0.99)
n_short = int((ts < 60).sum())
n_long = int((ts > p99).sum())

print(f"Short trips (<60s): {n_short:,} ({n_short/len(ts)*100:.2f}%)")
print(f"Long trips (>{p99:.0f}s): {n_long:,} ({n_long/len(ts)*100:.2f}%)")
print(f"Zero-duration trips: {(ts == 0).sum():,}")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(ts, bins=200, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0, 0].axvline(60, color="red", linestyle="--", label="< 60s")
axes[0, 0].axvline(p99, color="orange", linestyle="--", label=f"> {p99:.0f}s (p99)")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("Trip Duration - Full (log y)")
axes[0, 0].set_xlabel("Seconds")
axes[0, 0].legend()

short_ts = ts[ts < 60]
axes[0, 1].hist(short_ts, bins=60, color="red", edgecolor="white")
axes[0, 1].set_title(f"Short Trips (<60s) - n={n_short:,}")
axes[0, 1].set_xlabel("Seconds")

long_ts = ts[ts > p99]
axes[1, 0].hist(long_ts, bins=100, color="orange", edgecolor="white")
axes[1, 0].set_title(f"Long Trips (>{p99:.0f}s) - n={n_long:,}")
axes[1, 0].set_xlabel("Seconds")

bp_data = ts[(ts > 0) & (ts < ts.quantile(0.999))]
axes[1, 1].boxplot(bp_data, vert=False)
axes[1, 1].set_title("Trip Duration Boxplot (0 to p99.9)")
axes[1, 1].set_xlabel("Seconds")

fig.suptitle("Outlier Diagnostics: Trip Duration", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2b: Extreme Distances & Fares
- Very high mileage: `trip_miles > 99th percentile`
- Very low fare: `fare_usd < $0.50`

In [ ]:
miles = df["trip_miles"].dropna()
fare = df["fare_usd"].dropna()
miles_p99 = miles.quantile(0.99)
fare_p99 = fare.quantile(0.99)
low_fare_mask = fare < 0.50

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(miles, bins=200, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0, 0].axvline(miles_p99, color="red", linestyle="--", label=f"p99={miles_p99:.1f}")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("Trip Miles - Full (log y)")
axes[0, 0].set_xlabel("Miles")
axes[0, 0].legend()

extreme_miles = miles[miles > miles_p99]
axes[0, 1].hist(extreme_miles, bins=100, color="red", edgecolor="white")
axes[0, 1].set_title(f"Extreme Miles (>{miles_p99:.1f}) - n={len(extreme_miles):,}")
axes[0, 1].set_xlabel("Miles")

axes[1, 0].hist(fare, bins=200, color="darkorange", edgecolor="white", linewidth=0.3)
axes[1, 0].axvline(0.50, color="red", linestyle="--", label="< $0.50")
axes[1, 0].axvline(fare_p99, color="orange", linestyle="--", label=f"p99={fare_p99:.1f}")
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Fare USD - Full (log y)")
axes[1, 0].set_xlabel("USD")
axes[1, 0].legend()

low_fare = df[df["fare_usd"] < 0.50]
axes[1, 1].hist(low_fare["trip_seconds"].dropna(), bins=100, color="red", edgecolor="white")
axes[1, 1].set_title(f"Very Low Fare (<$0.50) Trip Duration - n={len(low_fare):,}")
axes[1, 1].set_xlabel("Seconds")

fig.suptitle("Outlier Diagnostics: Distance & Fare", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Extreme distances (>{miles_p99:.1f} mi): {len(extreme_miles):,}")
print(f"Very low fare (<$0.50): {low_fare_mask.sum():,}")
print(f"Zero distance: {(miles == 0).sum():,}")
print(f"Zero fare: {(fare == 0).sum():,}")

### 2c: Geographic Outliers - Outside Chicago Bounding Box
Chicago approximate bounds:
- Latitude: 41.60 - 42.05
- Longitude: -87.95 - -87.50

In [ ]:
CHI_LAT_MIN, CHI_LAT_MAX = 41.60, 42.05
CHI_LON_MIN, CHI_LON_MAX = -87.95, -87.50

pup_out = (
    (df["pickup_lat"] < CHI_LAT_MIN) | (df["pickup_lat"] > CHI_LAT_MAX)
    | (df["pickup_lon"] < CHI_LON_MIN) | (df["pickup_lon"] > CHI_LON_MAX)
)
dof_out = (
    (df["dropoff_lat"] < CHI_LAT_MIN) | (df["dropoff_lat"] > CHI_LAT_MAX)
    | (df["dropoff_lon"] < CHI_LON_MIN) | (df["dropoff_lon"] > CHI_LON_MAX)
)

n_pup_out = int(pup_out.sum())
n_dof_out = int(dof_out.sum())
print(f"Pickup outside Chicago: {n_pup_out:,} ({n_pup_out/len(df)*100:.2f}%)")
print(f"Dropoff outside Chicago: {n_dof_out:,} ({n_dof_out/len(df)*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sample_geo = df.sample(min(100_000, len(df)), random_state=42)

axes[0].scatter(sample_geo["pickup_lon"], sample_geo["pickup_lat"], s=1, alpha=0.3, c="steelblue")
rect = plt.Rectangle((CHI_LON_MIN, CHI_LAT_MIN), CHI_LON_MAX - CHI_LON_MIN, CHI_LAT_MAX - CHI_LAT_MIN,
                      fill=False, edgecolor="red", linewidth=2, linestyle="--", label="Chicago bbox")
axes[0].add_patch(rect)
axes[0].set_title("Pickup Locations (sample)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].legend()
axes[0].set_aspect("equal")

axes[1].scatter(sample_geo["dropoff_lon"].dropna(), sample_geo["dropoff_lat"].dropna(), s=1, alpha=0.3, c="darkorange")
rect2 = plt.Rectangle((CHI_LON_MIN, CHI_LAT_MIN), CHI_LON_MAX - CHI_LON_MIN, CHI_LAT_MAX - CHI_LAT_MIN,
                       fill=False, edgecolor="red", linewidth=2, linestyle="--", label="Chicago bbox")
axes[1].add_patch(rect2)
axes[1].set_title("Dropoff Locations (sample)")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].legend()
axes[1].set_aspect("equal")

fig.suptitle("Geographic Distribution & Chicago Bounding Box", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2d: Consistency Check - Trip Total vs Fare + Tips + Tolls + Extras
Verify that `trip_total_usd == fare_usd + tips_usd + tolls_usd + extras_usd`.

In [ ]:
components = ["fare_usd", "tips_usd", "tolls_usd", "extras_usd"]
valid_mask = df[components + ["trip_total_usd"]].notna().all(axis=1)

df_valid = df[valid_mask].copy()
df_valid["computed_total"] = df_valid[components].sum(axis=1)
df_valid["diff"] = df_valid["trip_total_usd"] - df_valid["computed_total"]
df_valid["diff_abs"] = df_valid["diff"].abs()

mismatch = df_valid[df_valid["diff_abs"] > 0.01]
pct_mm = len(mismatch) / len(df_valid) * 100
print(f"Rows with all components: {len(df_valid):,}")
print(f"Mismatches (|diff| > $0.01): {len(mismatch):,} ({pct_mm:.2f}%)")
print(f"Mean difference: ${df_valid['diff'].mean():.4f}")
print(f"Median difference: ${df_valid['diff'].median():.4f}")
print(f"Max positive diff: ${df_valid['diff'].max():.2f}")
print(f"Max negative diff: ${df_valid['diff'].min():.2f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_valid["diff"], bins=200, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title("Total - (Fare+Tips+Tolls+Extras)")
axes[0].set_xlabel("Difference (USD)")

zoom = df_valid["diff"][(df_valid["diff"] > -5) & (df_valid["diff"] < 5)]
axes[1].hist(zoom, bins=200, color="darkorange", edgecolor="white", linewidth=0.3)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Difference (zoom +-$5)")
axes[1].set_xlabel("Difference (USD)")

s = df_valid.sample(min(50_000, len(df_valid)), random_state=42)
max_val = min(s["trip_total_usd"].quantile(0.99), s["computed_total"].quantile(0.99))
axes[2].scatter(s["computed_total"], s["trip_total_usd"], s=1, alpha=0.3)
axes[2].plot([0, max_val], [0, max_val], "r--", label="Perfect match")
axes[2].set_xlabel("Fare + Tips + Tolls + Extras")
axes[2].set_ylabel("Trip Total")
axes[2].set_title("Computed vs Reported Total")
axes[2].legend()

fig.suptitle("Consistency Check: Trip Total vs Components", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 2e: Outlier Summary Table

In [ ]:
same_area_n = int((df["pickup_community_area"] == df["dropoff_community_area"]).sum())
summary = {
    "Category": [
        "Short trips (<60s)", "Zero-duration trips", "Very long trips (>p99)",
        "Zero distance", "Extreme distance (>p99)",
        "Zero fare", "Very low fare (<$0.50)",
        "Pickup == Dropoff area",
        "Pickup outside Chicago", "Dropoff outside Chicago",
        "Total != Components",
    ],
    "Count": [
        n_short, int((ts == 0).sum()), n_long,
        int((miles == 0).sum()), int(len(extreme_miles)),
        int((fare == 0).sum()), int(low_fare_mask.sum()),
        same_area_n, n_pup_out, n_dof_out,
        int(len(mismatch)),
    ]
}
summary_df = pd.DataFrame(summary)
summary_df["Pct"] = (summary_df["Count"] / len(df) * 100).round(2)
print(summary_df.to_string(index=False))